<a href="https://colab.research.google.com/github/nakamura196/NDLOCR-GoogleColabVersion/blob/main/NDLOCRv2_googlecolabversion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 0. GPUの情報を確認する

In [1]:
!nvidia-smi

Tue May 27 22:29:37 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip list | grep torch

torch                                 2.6.0+cu124
torchao                               0.10.0
torchaudio                            2.6.0+cu124
torchdata                             0.11.0
torchsummary                          1.5.1
torchtune                             0.6.1
torchvision                           0.21.0+cu124


In [20]:
# 中村追加
!pip install pip==24.0
!pip install numpy==1.26.4
!pip install --trusted-host download.pytorch.org torch==2.1.1 torchvision==0.16.1 torchaudio==2.1.1 torchtext==0.16.1 --index-url https://download.pytorch.org/whl/cu121

DEPRECATION: pytorch-lightning 1.6.5 has a non-standard dependency specifier torch>=1.8.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pytorch-lightning or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063
  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
DEPRECATION: pytorch-lightning 1.6.5 has a non-standard dependency specifier torch>=1.8.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pytorch-lightning or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063
  Attempting uninstall: numpy
    F

# 1. NDLOCRのリポジトリをcloneする(--recursiveを忘れずに！)

In [4]:
!git clone --recursive https://github.com/ndl-lab/ndlocr_cli -b feature/colab

Cloning into 'ndlocr_cli'...
remote: Enumerating objects: 545, done.
remote: Counting objects: 100% (119/119), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 545 (delta 75), reused 38 (delta 34), pack-reused 426 (from 2)
Receiving objects: 100% (545/545), 216.54 KiB | 1.13 MiB/s, done.
Resolving deltas: 100% (296/296), done.
Submodule 'submodules/deskew_HT' (https://github.com/ndl-lab/deskew_HT) registered for path 'submodules/deskew_HT'
Submodule 'submodules/ndl_layout' (https://github.com/ndl-lab/ndl_layout) registered for path 'submodules/ndl_layout'
Submodule 'submodules/reading_order' (https://github.com/ndl-lab/reading_order) registered for path 'submodules/reading_order'
Submodule 'submodules/ruby_prediction' (https://github.com/ndl-lab/ruby_prediction) registered for path 'submodules/ruby_prediction'
Submodule 'submodules/separate_pages_mmdet' (https://github.com/ndl-lab/separate_pages_mmdet) registered for path 'submodules/separate_pages_mmdet'
Submodule 

# 2. 必要なパッケージをインストールする

In [5]:
PROJECT_DIR="/content/ndlocr_cli"

In [9]:
# 中村追加
file_path = "/content/ndlocr_cli/requirements.txt"

# ファイル読み込み
with open(file_path, "r") as f:
    content = f.read()

# 修正処理
if "numpy==1.23.1" in content:
    content = content.replace("numpy==1.23.1", "numpy==1.23.2")

    # 修正内容を保存
    with open(file_path, "w") as f:
        f.write(content)

    print("✅ numpy のバージョンを更新しました。")
else:
    print("ℹ️ 修正対象の 'numpy==1.23.1' が見つかりません。既に修正済みかも。")

✅ numpy のバージョンを更新しました。


In [10]:
!pip install -r {PROJECT_DIR}/requirements.txt
#!pip install torch==2.0.0+cu118 torchvision==0.15.1+cu118 torchaudio==2.0.1 --index-url https://download.pytorch.org/whl/cu118
# !pip install mmcv==2.0.0 -f https://download.openmmlab.com/mmcv/dist/cu118/torch2.0/index.html

!pip install mmcv==2.1.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.1/index.html
!pip install torchmetrics==0.11.4

  Using cached evaluate-0.4.3-py3-none-any.whl.metadata (9.2 kB)
  Using cached hydra_colorlog-1.2.0-py3-none-any.whl.metadata (949 bytes)
  Using cached hydra_core-1.2.0-py3-none-any.whl.metadata (4.0 kB)
  Using cached lmdb-1.2.1.tar.gz (881 kB)
  Preparing metadata (setup.py) ... done
  Using cached lxml-4.9.3-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (3.8 kB)
  Using cached mmpretrain-1.2.0-py2.py3-none-any.whl.metadata (20 kB)
  Using cached nltk-3.6.2-py3-none-any.whl.metadata (2.9 kB)
  Using cached opencv_python-4.6.0.66-cp36-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (18 kB)
  Using cached pandas-1.5.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (11 kB)
  Using cached pyrootutils-1.0.4-py3-none-any.whl.metadata (4.0 kB)
  Using cached pytorch_lightning-1.6.5-py3-none-any.whl.metadata (32 kB)
  Using cached scikit_learn-1.2.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (11 kB)
  Using cached colorlog-6.9.0-p

Looking in links: https://download.openmmlab.com/mmcv/dist/cu121/torch2.1/index.html
DEPRECATION: pytorch-lightning 1.6.5 has a non-standard dependency specifier torch>=1.8.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pytorch-lightning or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063
DEPRECATION: pytorch-lightning 1.6.5 has a non-standard dependency specifier torch>=1.8.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pytorch-lightning or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063


In [11]:
'''
%cd {PROJECT_DIR}/
!wget http://www.phontron.com/kytea/download/kytea-0.4.7.tar.gz
!tar xzvf kytea-0.4.7.tar.gz
%cd kytea-0.4.7
!./configure
!make
!make install
!ldconfig
%cd /content
'''

'\n%cd {PROJECT_DIR}/\n!wget http://www.phontron.com/kytea/download/kytea-0.4.7.tar.gz\n!tar xzvf kytea-0.4.7.tar.gz\n%cd kytea-0.4.7\n!./configure\n!make\n!make install\n!ldconfig\n%cd /content\n'

In [ ]:
# !pip install kytea==0.1.7

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 74.6 MB/s eta 0:00:00


In [ ]:
#!pip install pytorch-lightning==2.0.4

In [12]:
%cd {PROJECT_DIR}/submodules/ndl_layout
# !git clone https://github.com/open-mmlab/mmdetection.git -b v3.0.0
!git clone https://github.com/open-mmlab/mmdetection.git -b v3.3.0
%cd {PROJECT_DIR}/submodules/ndl_layout/mmdetection
#下行はGPUのメモリ不足になった場合にコメントアウトを外すとよい。
#!sed -i -e 's/GPU_MEM_LIMIT = 1024\*\*3/GPU_MEM_LIMIT = 1024\*\*3\/\/5/' mmdet/models/roi_heads/mask_heads/fcn_mask_head.py
!python setup.py bdist_wheel
!pip install dist/*.whl
%cd /content

ストリーミング出力は最後の 5000 行に切り捨てられました。
copying mmdet/models/task_modules/__init__.py -> build/lib/mmdet/models/task_modules
copying mmdet/models/task_modules/builder.py -> build/lib/mmdet/models/task_modules
creating build/lib/mmdet/models/dense_heads
copying mmdet/models/dense_heads/boxinst_head.py -> build/lib/mmdet/models/dense_heads
copying mmdet/models/dense_heads/solo_head.py -> build/lib/mmdet/models/dense_heads
copying mmdet/models/dense_heads/pisa_ssd_head.py -> build/lib/mmdet/models/dense_heads
copying mmdet/models/dense_heads/fovea_head.py -> build/lib/mmdet/models/dense_heads
copying mmdet/models/dense_heads/rpn_head.py -> build/lib/mmdet/models/dense_heads
copying mmdet/models/dense_heads/detr_head.py -> build/lib/mmdet/models/dense_heads
copying mmdet/models/dense_heads/__init__.py -> build/lib/mmdet/models/dense_heads
copying mmdet/models/dense_heads/centernet_update_head.py -> build/lib/mmdet/models/dense_heads
copying mmdet/models/dense_heads/retina_head.py -> build/lib/mmde

# 4. OCRに必要な学習済みモデルをダウンロードする

In [13]:
%cd {PROJECT_DIR}
!wget -nc https://lab.ndl.go.jp/dataset/ndlocr_v2/text_recognition_lightning/resnet-orient2.ckpt -P ./submodules/text_recognition_lightning/models
!wget -nc https://lab.ndl.go.jp/dataset/ndlocr_v2/text_recognition_lightning/rf_author/model.pkl -P ./submodules/text_recognition_lightning/models/rf_author/
!wget -nc https://lab.ndl.go.jp/dataset/ndlocr_v2/text_recognition_lightning/rf_title/model.pkl -P ./submodules/text_recognition_lightning/models/rf_title/
!wget -nc https://lab.ndl.go.jp/dataset/ndlocr_v2/ndl_layout/ndl_retrainmodel.pth -P ./submodules/ndl_layout/models
!wget -nc https://lab.ndl.go.jp/dataset/ndlocr_v2/separate_pages_mmdet/epoch_180.pth -P ./submodules/separate_pages_mmdet/models
%cd /content/

/content/ndlocr_cli
--2025-05-27 22:39:57--  https://lab.ndl.go.jp/dataset/ndlocr_v2/text_recognition_lightning/resnet-orient2.ckpt
Resolving lab.ndl.go.jp (lab.ndl.go.jp)... 3.163.125.94, 3.163.125.48, 3.163.125.96, ...
Connecting to lab.ndl.go.jp (lab.ndl.go.jp)|3.163.125.94|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 575185573 (549M) [application/x-www-form-urlencoded]
Saving to: ‘./submodules/text_recognition_lightning/models/resnet-orient2.ckpt’

resnet-orient2.ckpt 100%[===================>] 548.54M  69.8MB/s    in 5.6s    

2025-05-27 22:40:03 (98.0 MB/s) - ‘./submodules/text_recognition_lightning/models/resnet-orient2.ckpt’ saved [575185573/575185573]

--2025-05-27 22:40:03--  https://lab.ndl.go.jp/dataset/ndlocr_v2/text_recognition_lightning/rf_author/model.pkl
Resolving lab.ndl.go.jp (lab.ndl.go.jp)... 3.163.125.94, 3.163.125.48, 3.163.125.96, ...
Connecting to lab.ndl.go.jp (lab.ndl.go.jp)|3.163.125.94|:443... connected.
HTTP request sent, await

# 5. PDFを画像に変換するためのパッケージのインストール

In [14]:
!apt-get install poppler-utils
!pip install pdf2image

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 35 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.8 [186 kB]
Fetched 186 kB in 1s (200 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 126109 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.8_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.8) ...
Setting up poppler-utils (22.02.0-2ubuntu0.8) ...
Processing triggers for man-db (2.10.2-1) ...
DEPRECATION: pytorch-lightning 1.6.5 has a non-standard dependency specifier torch>=1.8.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer versi

# 6. テキスト化したいPDFをダウンロードする
今回は、ROIS-DS人文学オープンデータ共同利用センター(http://codh.rois.ac.jp/
)が提供している

近代雑誌データセット　http://codh.rois.ac.jp/modern-magazine/
から、

東洋学芸雑誌(https://dglb01.ninjal.ac.jp/ninjaldl/bunken.php?title=toyogakuge)

第一号(https://dglb01.ninjal.ac.jp/ninjaldl/toyogakuge/001/PDF/tygz-001.pdf)

をダウンロードしてみます。

In [15]:
!curl https://dglb01.ninjal.ac.jp/ninjaldl/toyogakuge/001/PDF/tygz-001.pdf -o /content/tygz-001.pdf --insecure

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 14.1M  100 14.1M    0     0  8282k      0  0:00:01  0:00:01 --:--:-- 8281k


# 7. PDFをjpeg画像に変換する

In [16]:
from pathlib import Path
from pdf2image import convert_from_path
import os
pdf_path = Path("/content/tygz-001.pdf")
os.makedirs("/content/tygz-001/img",exist_ok=True)
img_path=Path("/content/tygz-001/img")

convert_from_path(pdf_path, output_folder=img_path,fmt='jpeg',output_file=pdf_path.stem,dpi=30)

[<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1112x1552>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1112x1552>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1112x1552>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1112x1552>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1112x1552>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1112x1552>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1112x1552>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1112x1552>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1112x1552>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1112x1552>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1112x1552>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1112x1552>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1112x1552>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1112x1552>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=R

# 8. OCRの実行

/content/tygz-001以下のimgディレクトリ内の画像を処理し、
/content/tygz-001_outputに出力する場合

In [18]:
%cd {PROJECT_DIR}
!python main.py infer /content/tygz-001 /content/tygz-001_output -s s -x

/content/ndlocr_cli
Traceback (most recent call last):
  File "/content/ndlocr_cli/main.py", line 11, in <module>
    from ocrcli.core import OcrInferrer, OcrResultEvaluator
  File "/content/ndlocr_cli/ocrcli/core/__init__.py", line 7, in <module>
    from .inference import OcrInferrer
  File "/content/ndlocr_cli/ocrcli/core/inference.py", line 18, in <module>
    from .. import procs
  File "/content/ndlocr_cli/ocrcli/procs/__init__.py", line 10, in <module>
    from .line_ocr import LineOcrProcess
  File "/content/ndlocr_cli/ocrcli/procs/line_ocr.py", line 5, in <module>
    import hydra
  File "/usr/local/lib/python3.11/dist-packages/hydra/__init__.py", line 5, in <module>
    from hydra import utils
  File "/usr/local/lib/python3.11/dist-packages/hydra/utils.py", line 8, in <module>
    import hydra._internal.instantiate._instantiate2
  File "/usr/local/lib/python3.11/dist-packages/hydra/_internal/instantiate/_instantiate2.py", line 12, in <module>
    from hydra._internal.utils im

# 9. 結果の確認

In [19]:
import glob
import os
for fpath in sorted(glob.glob("/content/tygz-001_output/tygz-001/txt/*_main.txt")):
    with open(fpath) as f:
        txtdata=f.read()
        print(os.path.basename(fpath).replace("_main.txt",""))
        print(txtdata)